Core Mechanics & Theory
In CPython (the standard Python implementation written in C), variables do not hold values directly; they hold pointers to objects in memory.

1. The PyObject Structure
Every entity in Python is a C struct called PyObject (or PyVarObject for variable-length items like lists or strings). At its core, every object in memory contains at least two header fields:

ob_refcnt: Reference counter (used by the Garbage Collector).

ob_type: Pointer to the object’s type struct (e.g., int, str, list).

Variable 'a' (Pointer) ---> [ PyObject Header (refcnt, type) | Value Payload ]

2. Identity vs. Equality
id(obj): Returns the memory address of the object as an integer (in CPython, this is the actual virtual memory address of the PyObject pointer).

is: Checks pointer identity (id(a) == id(b)). Tests whether two variables point to the exact same block of memory.

==: Checks value equality. Invokes the object's __eq__() method to compare the contents.

In [4]:
a=1
print(id(a))
type(a)

2007148396784


int

In [12]:
import sys

In [13]:
def diagnose(obj1, obj2):
    print(f"memory of the obj1: {hex(id(obj1))} \n and memory of the object 2 {hex(id(obj2))}")
    print(f"do they share same memory?{obj1 is obj2}")
    print(f"do they match the values{obj1==obj2}")
    print(f"refference count of obj1 :{sys.getrefcount(obj1)} \n and for the obj2 {sys.getrefcount(obj2)}")

In [14]:
diagnose(129,128)

memory of the obj1: 0x1d3535e10f0 
 and memory of the object 2 0x1d3535e10d0
do they share same memory?False
do they match the valuesFalse
refference count of obj1 :69 
 and for the obj2 157


In [15]:
diagnose(129,129)

memory of the obj1: 0x1d3535e10f0 
 and memory of the object 2 0x1d3535e10f0
do they share same memory?True
do they match the valuesTrue
refference count of obj1 :72 
 and for the obj2 72


#### created during runtime doesnt assign

In [ ]:
int("128") is int("128") #only from -5 to 256

True

In [23]:
diagnose(500,500)
int("500") is int("500")

memory of the obj1: 0x1d359864490 
 and memory of the object 2 0x1d359864490
do they share same memory?True
do they match the valuesTrue
refference count of obj1 :7 
 and for the obj2 7


False

In [16]:
diagnose("dad","dad")

memory of the obj1: 0x1d359903cb0 
 and memory of the object 2 0x1d359903cb0
do they share same memory?True
do they match the valuesTrue
refference count of obj1 :7 
 and for the obj2 7


In [18]:
def dynamic(x):
    return x*1

x=0
while dynamic(x) is x:
    x-=1
print("lower boundary is ",x+1)

x=0
while dynamic(x) is x:
    x+=1
print("upper boundary is ",x-1)

lower boundary is  -5
upper boundary is  256


sys.intern() tells Python to reuse one shared string object for identical strings.

In [21]:
s1= "".join(["hello","world","!"])
s2= "".join(["hello","world","!"])

print(s1==s2)
print(s1 is s2)

s1_in= sys.intern(s1)
s2_in= sys.intern(s2)
print(s1_in is s2_in)

True
False
True


## Mutability, References & Copies
Core Mechanics & Theory
Understanding mutability at the memory level is crucial for building custom data structures and managing tensor memory layouts.

1. Mutability vs. Immutability Under the Hood
Immutable Types (int, float, str, tuple, frozenset, bytes): Once allocated in memory, their payload cannot change. Any "modification" allocates a completely new PyObject with a new id() and updates the variable pointer.

Mutable Types (list, dict, set, bytearray): The PyObject maintains a fixed memory address, but points to internal data buffers (like arrays of pointers) that can be resized or modified in place.

2. The Reference Trap: Reassignment vs. In-Place Mutation
Reassignment (a = a + [1]): Allocates a brand new list object and points a to the new address.

In-place mutation (a.append(1) or a += [1]): Modifies the existing list at its current memory address via __iadd__. Any other variable referencing the same id sees the mutation.

3. Shallow Copy vs. Deep Copy
Shallow Copy (copy.copy(), list.copy(), [:]): Allocates a new outer container object, but populates it by copying the pointers to the original inner elements.

Modifying the top-level structure of the copy does not affect the original.

Modifying a nested mutable object inside the copy does affect the original.

Deep Copy (copy.deepcopy()): Recursively creates new container objects and copies all elements down the entire object graph, maintaining a memoization dictionary (memo) to handle cyclic references (e.g., an object pointing to itself).

In [ ]:
def test_mutation():
    l1= [1,2,3]
    l2=l1
    l1+=[4]
    
    print(l1 is l2)#mutable
    
    tup1=(1,2,3)
    tup2=tup1
    
    tup1+=(4,)
    print(tup1 is tup2)#immutable creates new obj
test_mutation()

True
False


In [28]:
matrix = [[0]*3]*3
matrix

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [29]:
matrix[0][0]=1
matrix

[[1, 0, 0], [1, 0, 0], [1, 0, 0]]

In [30]:
for i in range(3):
    print(id(matrix[i][0]),"/n")

2007148396784 /n
2007148396784 /n
2007148396784 /n


In [31]:
for i in range(3):
    print(id(matrix[0][i]),"/n")

2007148396784 /n
2007148396752 /n
2007148396752 /n


In [50]:
def deep_copy(a,memo=None):
    if memo is None:
        memo={}
    if isinstance(a,(int,float,str,bool)) or a is None:
        return a
    
    old_id=id(a)
    if old_id in memo:
        return memo[old_id]
    
    #list
    if isinstance(a,list):
        new_obj=[]
        memo[old_id]=new_obj
        
        for item in a:
            new_obj.append(deep_copy(item,memo))
        return new_obj
    #dict
    if isinstance(a,dict):
        new_obj={}
        memo[old_id]=new_obj
        for key, val in a.items():
            new_key=deep_copy(key)
            new_val = deep_copy(val)
            new_obj[new_key]=new_val
        return new_obj 
    if isinstance(a, set):
        new_obj = set()
        memo[old_id] = new_obj

        for item in a:
            new_obj.add(deep_copy(item, memo))

        return new_obj

    # Tuple
    if isinstance(a, tuple):
        new_obj = tuple(deep_copy(item, memo) for item in a)
        memo[old_id] = new_obj

        return new_obj

    raise TypeError(f"Unsupported type: {type(a)}")
    

In [51]:
a = [1, 2]
a.append(a)  # Circular reference
nested = {"key": [1, (2, 3), {4, 5}], "self": a}
cloned = deep_copy(nested)

In [53]:
cloned is not nested

True

In [54]:
cloned["key"][0] == nested["key"][0]


True

In [55]:
cloned["key"][1] is nested["key"][1] 


False

In [56]:

cloned["self"][2] is cloned["self"] 

True

Truth Value Testing (Truthy vs. Falsy)
Under the hood, when Python evaluates if x:, it checks the following CPython sequence:

Calls x.__bool__() if defined (must return True or False).

If __bool__() is not defined, calls x.__len__(). If it returns 0, the object is Falsy; otherwise, Truthy.

If neither is defined, all objects default to Truthy.

Built-in Falsy objects: None, False, 0, 0.0, 0j, Decimal(0), Fraction(0, 1), "", (), [], {}, set(), range(0).

In [61]:
for i in (None, False, 0, 0.0, 0j,"", (), [], {}, set()):
    if i:
        print("it will not print this")

Short-Circuit Evaluation (and / or) Returns Operands, Not Booleans
Python logical operators do not strictly return boolean True/False; they return the actual operand that determined the expression's truth value:

X and Y: Evaluates X. If X is falsy, immediately returns X (short-circuits without evaluating Y). Otherwise, evaluates and returns Y.

X or Y: Evaluates X. If X is truthy, immediately returns X (short-circuits without evaluating Y). Otherwise, evaluates and returns Y.

3. The Walrus Operator (:= Assignment Expressions)
Introduced in Python 3.8, NAME := expr evaluates expr, assigns the value to NAME, and returns the value in place.

Solves the common anti-pattern of computing or fetching a value twice (once for the conditional check and once inside the block).

Scoping: The variable assigned via := leaks into the surrounding function/module scope (it is not localized to the if/while block).

In [62]:
name = input("Enter your name: ")

while name != "quit":
    print("Hello", name)
    name = input("Enter your name: ")

Hello hi
Hello quite


In [65]:
while (name:=input("enter your name"))!='quit':
    print("name is",name)

name is ji


Custom Object Truthiness Engine
Create a class DataPacket that accepts a payload (list or None) and an integer status_code.

Implement __len__ and __bool__ methods such that:

A packet is considered Truthy only if status_code == 200 AND the payload contains at least one item.

If status_code != 200, the packet must evaluate to Falsy even if the payload has items.

Verify its behavior in conditional branching (if packet: ...) across various status codes and payload sizes.

In [69]:
class DataPacket:
    def __init__(self,payload, status_code):
        self.payload=payload
        self.status_code=status_code

    def __len__(self):
        if self.payload is None:
            return 0
        return len(self.payload)
    
    def __bool__(self):
        return self.status_code == 200 and len(self) > 0

In [70]:
p1 = DataPacket([1, 2, 3], 200)
p2 = DataPacket([], 200)
p3 = DataPacket([1, 2, 3], 404)
p4 = DataPacket(None, 200)

print(bool(p1))  # True
print(bool(p2))  # False
print(bool(p3))  # False
print(bool(p4))  # False

True
False
False
False


Exercise 2: Short-Circuit Pipeline Evaluator
Write a pure short-circuit expression (no if-else keywords) that takes a dictionary config = {} and extracts a valid port number using the following precedence:

config["override_port"] (if positive integer)

config["default_port"] (if positive integer)

Fallback to 8080

Constraint: Write this as a single-line assignment using only or, and, dictionary access methods, and comparison checks.

## condition and value or fallback

In [80]:
def constraint(config: dict=None):
    port=config["override_port"]>=0 and config["override_port"] or config["default_port"]>=0 and config["default_port"] or 8080
    print(port)

In [81]:
constraint({"override_port":-1,"default_port":-1})

8080


In [82]:
constraint({"override_port":0,"default_port":90})

90


Exercise 3: Stream Tokenizer with Walrus (:=)
Write a string parser function parse_tokens(raw_string: str) that extracts and yields numbers formatted as [#123], [#456], etc., using a while loop with a single regex search or string slicing index.

Requirements:

Use assignment expressions (:=) inside your while loop condition to find the next match index / pattern without calling the search function twice.

Extract only the digits into an integer list.

Example input: "Log 01: [#42] succeeded. Log 02: [#108] failed. Log 03: [#999] pending."

Expected output: [42, 108, 999]

In [1]:
import re

def parse_tokens(raw_string: str):
    pattern = re.compile(r"\[#(\d+)\]")
    numbers = []
    pos = 0

    while (match := pattern.search(raw_string, pos)):
        numbers.append(int(match.group(1)))
        pos = match.end()

    return numbers

In [5]:
#alternatively
import re

def parse_tokens_findall(raw_string: str):
    return [int(x) for x in re.findall(r"\[#(\d+)\]", raw_string)]

In [2]:
raw = "Log 01: [#42] succeeded. Log 02: [#108] failed. Log 03: [#999] pending."

print(parse_tokens(raw))

[42, 108, 999]


In [6]:
raw = "Log 01: [#42] succeeded. Log 02: [#108] failed. Log 03: [#999] pending."

print(parse_tokens_findall(raw))

[42, 108, 999]
